In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np

import sys
sys.path.append('../') 
from core.layer import Dense

# Part 1: Mathematical Derivations
The goal of this notebook:
* **Rigorous Proof:** Go deeper in the core of **Backward propagation** algorithm and **Chain-Rule**
* **Empirical Verification:** Some tests to proof that the mathematical theory is match to the computer logic

## 1. Mathmetics of Backpropagation (Dense Layer)
In this section, we will make clear the mathmetics formula behind the `backward` function of a Dense Layer. Understanding the dimesions of the matries is the key to successfully building a Deep Learning Framework

### 1.1. Notation and Vector Space
Define some dimesions:
* $N$: Number of input data samples
* $I$: Number of input data features (Input neurons)
* $O$: Number of output neurons

The shapes of matrix in **Foward** are:
* Input matrix $X \in \mathbb{R}^{N \times I}$
* Weight matrix $W \in \mathbb{R}^{I \times O}$
* Bias matrix $b \in \mathbb{R}^{1 \times O}$

The linear output $Z$ is calculated as:
$$Z = X \cdot W +b \quad \rightarrow \quad Z \in \mathbb{R}^{N \times O}$$

### 1.2. Matrix Calculus in the Backward
Let $L$ is the general loss fuction. In Backward Pass, the Dense Layer receives **Derivative L with respect to Z** from subsequent layer. This is denote as:
$$\frac{\partial L}{\partial Z} \in \mathbb{R}^{N \times O}$$
* In the code, this matrix corresponds to the `output_error` variable

#### A. Derivative with respect to Weights $W$ (To update $W$)
To find $L$ changes as $W$ changes. We use formula:
$$\frac{\partial L}{\partial W}=X^T \cdot \frac{\partial L}{\partial Z} \in \mathbb{R}^{I \times O}$$
Proving above function:

Let's isolate a single weight $W_{kj}$ (the weight connection $k$-th feature to $j$-th output neuron)

Recall the foward pass function for single element $Z_{ij}$:
$$Z_{ij}=W_{1j} \times X_{i1} + W_{2j} \times X_{i2} + \cdots + W_{kj} \times X_{ik} +\cdots + W_{Ij} \times X_{iI} + b_{j}$$
Taking the partial derivative of $Z_{ij}$ with respect to $W_{kj}$:
$$\frac{\partial Z_{ij}}{\partial W_{kj}}=X_{ik}$$

Substituting this result back into chain rule equation
$$\frac{\partial L}{\partial W_{kj}} = \sum_{i=1}^{N} \left( \frac{\partial L}{\partial Z_{ij}} \cdot \frac{\partial Z_{ij}}{\partial W_{kj}} \right) = \sum_{i=1}^{N} \left( X_{ik} \cdot \frac{\partial L}{\partial Z_{ij}} \right)$$

**The Matrix Multiplication:**
* $X_{ik}$ is the element at row $i$ and column $k$ of matrix $X$ and can be defined as row $k$ and column $i$ of matrix $X^T$
* The summation id calculating the dot product between row $k$ of $X^T$ and column $j$ of $\frac{\partial L}{\partial Z}$
* By the fundamental definition of matrix multiplication, this operation constructs the element at position $(k, j)$ of the resulting matrix.

#### B. Derivative with respect to Input $X$ (To propagate error to previous layer)
To find $L$ changes as the contribution of Input $X$. We use formula:
$$\frac{\partial L}{\partial X}=W^T \cdot \frac{\partial L}{\partial Z} \in \mathbb{R}^{N \times I}$$

#### C. Derivative with respect to bias $b$ (To update $b$)
$$\frac{\partial L}{\partial b}=\sum_{i=1}^{N}\frac{\partial L}{\partial Z_i} \in \mathbb{R}^{1 \times O}$$

## 2. Test Dense Layer

### 2.1. Define Dense Layer

In [4]:
my_layer=Dense(input_size=3,output_size=2)

In [5]:
print(my_layer.weights)
print(f'Shape of weights matrix: {my_layer.weights.shape}')

[[0.77188538 0.56321283]
 [0.56086685 0.29864605]
 [0.27620453 0.40536581]]
Shape of weights matrix: (3, 2)


In [6]:
print(my_layer.bias)
print(f'Shape of weights matrix: {my_layer.bias.shape}')

[[0.32304928 0.92983841]]
Shape of weights matrix: (1, 2)


### 2.2. Test foward propagation

In [7]:
X= np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
    [0.5, 1.5, 2.5]
])

In [8]:
X.shape

(4, 3)

In [10]:
Z=my_layer.forward(X)

In [11]:
print(Z)
print(f'Shape of output: {Z.shape}')

[[ 3.04528194  3.30644075]
 [ 7.87215221  7.10811479]
 [12.69902249 10.90978883]
 [ 2.24080356  2.67282841]]
Shape of output: (4, 2)


### 2.3. Test back propagation

**Hypothetical output error matrix receives by next layer**

In [12]:
mock_output_error = np.array([
    [0.1, -0.2],
    [-0.1, 0.3],
    [0.5, 0.1],
    [-0.2, 0.4]
])
learning_rate=0.01

In [13]:
mock_output_error.shape

(4, 2)

**Calculate input error**

In [14]:
input_error=my_layer.backward(mock_output_error,learning_rate)
input_error

array([[-0.03545403, -0.00364252, -0.05345271],
       [ 0.09177531,  0.03350713,  0.09398929],
       [ 0.44226397,  0.31029803,  0.17863884],
       [ 0.07090805,  0.00728505,  0.10690542]])

In [15]:
print(f'Shape of input error: {input_error.shape}')

Shape of input error: (4, 3)


**Weights after learning**

In [16]:
my_layer.weights

array([[0.74088538, 0.54421283],
       [0.52686685, 0.27364605],
       [0.23920453, 0.37436581]])

**Bias after learning**

In [17]:
my_layer.bias

array([[0.32004928, 0.92383841]])